# FBref — estatísticas complementares da La Liga, 2017–2025

O que o Understat não tem — hoje, só o **básico**: desarmes ganhos,
interceptações, faltas cometidas e sofridas, cruzamentos, cartões.

> **O FBref perdeu os dados avançados da Opta** (checado em set/2026, todas as
> temporadas 2017–2025): as colunas de desarmes por terço, bloqueios, cortes,
> duelos aéreos, recuperações, progressão e PSxG continuam no cabeçalho, mas as
> células vêm vazias. Este notebook descarta sozinho as colunas 100% vazias.
> Antes de baixar um tipo de tabela novo, salve uma temporada e confira.

**Por que uma janela do Chrome.** O FBref fica atrás de um desafio do
Cloudflare que barra requisições automáticas — `requests` e Chrome invisível
recebem só "Um momento…". Este notebook abre uma janela **visível** do seu
Chrome: se a verificação aparecer, você clica nela uma vez, como faria
navegando. Depois disso, a mesma sessão percorre as páginas em ritmo humano
(uma a cada ~7 s — o FBref bane acima de ~10 requisições por minuto).

- O HTML cru de cada página fica em `data/_fbref/html/` — rodar de novo só
  busca o que falta.
- O perfil do navegador fica em `data/_fbref/browser_profile/`, então a
  liberação do Cloudflare costuma valer para as próximas execuções.
- Saída: `data/fbref_stats.parquet`, uma linha por (temporada, jogador, clube).

> **Licença.** Os dados avançados do FBref vêm da Opta. Uso em protótipo ok;
> num produto vendido a clubes, é preciso um feed licenciado.

**Rode com você na frente do computador** — a janela aparece na tela.

In [ ]:
import os
import sys
import re
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup
from playwright.sync_api import sync_playwright


def find_project_root(marker: str = "CLAUDE.md") -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / marker).exists():
            return candidate
    raise RuntimeError(f"could not find {marker} at or above {here}")


BASE_PATH = Path(os.environ.get("TRANSFER_EDGE_ROOT") or find_project_root())
sys.path.insert(0, str(BASE_PATH))
FBREF_DIR = BASE_PATH / "data" / "_fbref"
HTML_DIR = FBREF_DIR / "html"
PROFILE_DIR = FBREF_DIR / "browser_profile"
HTML_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = BASE_PATH / "data" / "fbref_stats.parquet"
MATCHED_PATH = BASE_PATH / "data" / "fbref_matched.parquet"

YEARS = list(range(2017, 2026))   # 2017 = 2017-18, igual aos notebooks 01 e 02
# O FBref barra navegação automática (Cloudflare). Com AUTO_FETCH = False, você
# baixa as páginas à mão (seção 1 lista quais faltam); True tenta abrir o Chrome.
AUTO_FETCH = False
PAGE_DELAY = 7.0                  # segundos entre páginas: ~8,5/min, abaixo do limite
CHALLENGE_TIMEOUT = 180           # quanto esperar pelo seu clique no Cloudflare

# stat type na URL -> id da tabela de jogadores na página.
# Só "misc" ainda tem dado: desarmes ganhos, interceptações, faltas, cruzamentos,
# cartões. Checado em set/2026 nas 9 temporadas:
#   - defense: as colunas avançadas vêm vazias; as que sobram (tackles_won,
#     interceptions) são idênticas às de misc — redundante.
#   - possession, passing, keepersadv, playingtime (xG em campo): dados avançados
#     da Opta, removidos. Antes de acrescentar um tipo aqui, salve uma temporada e
#     confira que as colunas vêm preenchidas.
TABLES = {
    "misc": "stats_misc",
}


def page_url(stat: str, year: int) -> str:
    season = f"{year}-{year + 1}"
    return f"https://fbref.com/en/comps/12/{season}/{stat}/{season}-La-Liga-Stats"


def cache_path(stat: str, year: int) -> Path:
    return HTML_DIR / f"{year}_{stat}.html"


print(f"project root: {BASE_PATH}")
print(f"{len(TABLES)} tabelas x {len(YEARS)} temporadas = {len(TABLES) * len(YEARS)} páginas")

## 1. Páginas: download manual

Com `AUTO_FETCH = False` (o padrão), esta célula só lista as páginas que ainda não
estão em `data/_fbref/html/`. Para cada uma: abra a URL no Chrome, **Cmd+S → Formato
"Página da Web, somente HTML"**, salve com o nome indicado nessa pasta. Não precisa
baixar todas — as seções 2 e 3 usam as tabelas que estiverem lá.

### Busca automática (desligada)


Abre o Chrome, e para cada página ainda não baixada: carrega, espera o
Cloudflare liberar (se pedir verificação, **clique na janela**), confere que a
tabela está lá e grava o HTML. Se o FBref devolver a página de limite de
requisições (429), para na hora — continue depois de ~1 hora, o cache guarda o
que já veio.

In [ ]:
class FBrefBlocked(RuntimeError):
    """The FBref rate limit (429) page, or a challenge nobody cleared."""


def is_challenge(page) -> bool:
    title = page.title().lower()
    return any(t in title for t in ("just a moment", "um momento", "attention required"))


def fetch_page(page, url: str, table_id: str) -> str:
    """Load one page in the open browser and return its HTML once the table is there."""
    page.goto(url, wait_until="domcontentloaded", timeout=60_000)

    waited = 0
    while is_challenge(page):
        if waited == 0:
            print("  -> verificação do Cloudflare: clique na janela do Chrome")
        if waited >= CHALLENGE_TIMEOUT:
            raise FBrefBlocked(f"verificação não liberada em {CHALLENGE_TIMEOUT}s: {url}")
        time.sleep(2)
        waited += 2

    html = page.content()
    if "Rate Limited Request" in html or "429 error" in html:
        raise FBrefBlocked("FBref limitou as requisições (429). Espere ~1 hora e rode de novo.")
    if f'id="{table_id}"' not in html:
        raise FBrefBlocked(f"a página carregou sem a tabela {table_id}: {url}")
    return html


todo = [(stat, year) for year in YEARS for stat in TABLES if not cache_path(stat, year).exists()]
print(f"{len(TABLES) * len(YEARS) - len(todo)} páginas em cache, {len(todo)} a buscar "
      f"(~{len(todo) * PAGE_DELAY / 60:.0f} min)")

def fetch_all(todo: list) -> None:
    with sync_playwright() as p:
        browser = p.chromium.launch_persistent_context(
            str(PROFILE_DIR), channel="chrome", headless=False,
            viewport={"width": 1280, "height": 900},
        )
        page = browser.pages[0] if browser.pages else browser.new_page()
        try:
            for n, (stat, year) in enumerate(todo, 1):
                html = fetch_page(page, page_url(stat, year), TABLES[stat])
                cache_path(stat, year).write_text(html)
                print(f"  {n}/{len(todo)}  {year} {stat}", flush=True)
                if n < len(todo):
                    time.sleep(PAGE_DELAY)
        finally:
            browser.close()


if todo and not AUTO_FETCH:
    print(f"\nSalve em {HTML_DIR}:")
    for stat, year in todo:
        print(f"  {cache_path(stat, year).name:24} <- {page_url(stat, year)}")
elif todo:
    # Playwright's sync API refuses to run inside Jupyter's event loop; a worker
    # thread has no loop of its own, so the same code works here and in a script.
    with ThreadPoolExecutor(max_workers=1) as pool:
        pool.submit(fetch_all, todo).result()

print("ok")

## 2. Ler as tabelas

O FBref guarda boa parte das tabelas dentro de comentários HTML, então os
comentários são removidos antes de ler. Cada célula tem um atributo `data-stat`
com um nome estável (`tackles_won`, `progressive_carries`, …) — é ele que vira o
nome da coluna, em vez do cabeçalho de duas linhas da página.

Cada jogador é identificado pelo ID do FBref (o trecho de 8 caracteres no link
do jogador), não pelo nome.

In [ ]:
IDENTITY = ["player_name_fb", "nationality", "position", "team", "age", "birth_year"]


def parse_table(html: str, table_id: str) -> pd.DataFrame:
    """One row per player-club in a FBref league table, columns named by data-stat."""
    soup = BeautifulSoup(html.replace("<!--", "").replace("-->", ""), "html.parser")
    table = soup.find("table", id=table_id)
    rows = []
    for tr in table.select("tbody > tr"):
        if "thead" in (tr.get("class") or []):          # repeated header rows
            continue
        cell = tr.find(attrs={"data-stat": "player"})
        link = cell.find("a", href=True) if cell else None
        match = re.search(r"/players/([0-9a-f]{8})/", link["href"]) if link else None
        if not match:
            continue
        record = {"fbref_id": match.group(1), "player_name_fb": link.get_text(strip=True)}
        for td in tr.find_all(["td", "th"]):
            key = td.get("data-stat")
            if key and key not in ("ranker", "player", "matches"):
                record[key] = td.get_text(strip=True)
        rows.append(record)
    return pd.DataFrame(rows)


def check_season(html: str, year: int, path: Path) -> None:
    """A hand-saved page for the wrong season is easy to miss — the table looks right."""
    title = re.search(r"<title>(.*?)</title>", html, re.S)
    found = re.search(r"(20\d\d-20\d\d)", title.group(1)) if title else None
    expected = f"{year}-{year + 1}"
    if not found or found.group(1) != expected:
        raise ValueError(
            f"{path.name} é da temporada {found.group(1) if found else '?'}, esperado {expected}. "
            f"Baixe de novo: {page_url(path.stem.split('_', 1)[1], year)}"
        )


def season_frame(year: int) -> pd.DataFrame:
    """All tables of one season merged into one row per (player, club)."""
    merged = None
    for stat, table_id in TABLES.items():
        path = cache_path(stat, year)
        if not path.exists():
            continue
        html = path.read_text()
        check_season(html, year, path)
        df = parse_table(html, table_id)
        if df.empty:
            continue
        key = ["fbref_id", "team"]
        if merged is None:
            merged = df
        else:
            # several tables repeat the same columns (interceptions, minutes_90s,
            # identity fields) — keep the first copy only
            new = [c for c in df.columns if c not in merged.columns]
            merged = merged.merge(df[key + new], on=key, how="outer")
    if merged is None:
        return pd.DataFrame()
    merged.insert(0, "year", year)
    return merged


fbref = pd.concat([season_frame(y) for y in YEARS], ignore_index=True)

# everything that isn't an identifier is a number ("1,234" -> 1234, "" -> NaN)
text_cols = {"fbref_id", "player_name_fb", "nationality", "position", "team", "age"}
for col in fbref.columns.difference(list(text_cols) + ["year"]):
    fbref[col] = pd.to_numeric(fbref[col].astype(str).str.replace(",", ""), errors="coerce")

# FBref lost its Opta advanced data: the headers are still there but the cells
# are blank (tackles by third, blocks, clearances, progression, PSxG...). Drop
# every column that is empty across the whole panel so nothing dead reaches a model.
empty = [c for c in fbref.columns if fbref[c].isna().all() or (fbref[c].astype(str).str.strip() == "").all()]
fbref = fbref.drop(columns=empty)
print(f"colunas vazias descartadas ({len(empty)}): {empty}")
print(f"fbref: {fbref.shape[0]:,} linhas x {fbref.shape[1]} colunas")
fbref.groupby("year").size().rename("jogadores por temporada").to_frame().T

## 3. Conferir e salvar

Três checagens antes de gravar: a chave (temporada, jogador, clube) é única,
toda temporada tem um número plausível de jogadores, e as colunas-chave das
tabelas novas vieram preenchidas.

In [ ]:
key_dups = fbref.duplicated(subset=["year", "fbref_id", "team"]).sum()
per_season = fbref.groupby("year").size()
# the headline column of each table that was downloaded
# misc: FBref no longer ships aerial duels or ball recoveries (checked 2017-2025),
# only cards, fouls, offsides, crosses and penalties.
HEADLINE = {"defense": "tackles_won", "misc": "crosses", "possession": "progressive_carries",
            "passing": "progressive_passes", "playingtime": "games_starts", "keepersadv": "gk_psxg"}
present = [s for s in HEADLINE if s in TABLES and any(cache_path(s, y).exists() for y in YEARS)]
must_have = [HEADLINE[s] for s in present]

print(f"chave duplicada             : {key_dups}")
print(f"jogadores por temporada     : {per_season.min()}–{per_season.max()}")
print(f"tabelas baixadas            : {present}")
print(f"preenchimento dessas colunas:")
print((fbref[must_have].notna().mean() * 100).round(0).to_string())

assert key_dups == 0, "chave (year, fbref_id, team) duplicada"
assert per_season.between(400, 800).all(), f"temporada com número estranho de jogadores:\n{per_season}"
missing = [c for c in must_have if c not in fbref.columns]
assert not missing, f"tabela baixada mas coluna ausente: {missing} — layout do FBref mudou?"

fbref.to_parquet(OUT_PATH, index=False)
print(f"\nSalvo {OUT_PATH.name}: {fbref.shape}")
fbref.head()

## 4. Casar com o painel do Transfermarkt

Por temporada, como o join do Understat no notebook 02, mas com uma trava a mais:
quando os dois lados têm **ano de nascimento**, ele precisa ser igual. Com essa
trava o nome pode ser mais solto (score ≥ 70) — é o que casa "Emerson" com
"Emerson Royal" sem arriscar homônimos. Sem ano de nascimento, vale a regra do
notebook 02 (nome ≥ 85, clube confirma nome inexato). Transferências no meio da
temporada têm as passagens somadas, como no Understat.

Precisa do `panel_final.parquet` (notebook 02). `src/data/loader.py` junta o
resultado ao painel sozinho, e `build_features()` cria as features por 90.


In [ ]:
from src.data.fbref import match_to_panel

panel = pd.read_parquet(BASE_PATH / "data" / "panel_final.parquet")
matched = match_to_panel(panel, fbref)

assert not matched.duplicated(["year", "player_id"]).any(), "jogador do painel casado duas vezes"
assert not matched.duplicated(["year", "fbref_id"]).any(), "jogador do FBref casado duas vezes"

coverage = matched.groupby("year").size() / panel.assign(year=panel.year.astype(int)).groupby("year").size()
print(f"casados {len(matched):,} de {len(panel):,} linhas do painel ({len(matched) / len(panel):.0%})")
print("por temporada:", {int(k): f"{v:.0%}" for k, v in coverage.items()})

matched.to_parquet(MATCHED_PATH, index=False)
print(f"Salvo {MATCHED_PATH.name}: {matched.shape}")
